# ForecastEx - forcus on NYC Mayor group forecast

## Web REST API

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests, json, os
from pprint import pprint

## Get live markets

In [3]:
from get_live_markets import get_live_markets
markets = get_live_markets()

In [4]:
from add_category_to_markets import add_category_to_markets
df_markets = add_category_to_markets(markets)

In [5]:
df_markets = df_markets[['category', 'name', 'symbol', 'conid']].sort_values(by=['category', 'name'])

In [6]:
conid_to_market = {market['conid']: market for market in markets}

In [10]:
df_markets[df_markets.symbol == 'MNYCG']

,category,name,symbol,conid
0,Election,General Election for New York City Mayor,MNYCG,796056051


In [12]:
market = markets[0]
market

{'symbol': 'MNYCG',
 'conid': 796056051,
 'name': 'General Election for New York City Mayor'}

## Get forecastex markets excluding financials

In [13]:
from fetch_all_contracts import fetch_all_contracts
from tqdm.auto import tqdm

In [14]:
for market in tqdm(markets):
    if 'contracts' not in market:
        market['contracts'] = fetch_all_contracts(market['conid'])
        print(market['conid'], len(market['contracts']))
    break

  0%|          | 0/211 [00:00<?, ?it/s]

796056051 10


In [20]:
forecastex = [market] #  for market in markets if len(market['contracts']) > 0]

In [21]:
len(markets), len(forecastex)

(211, 1)

## Get current probability for each market contract

In [22]:
from get_candidate_probability import get_candidate_probability

In [23]:
from tqdm.auto import tqdm

In [24]:
for market in tqdm(forecastex):
    for contract in market['contracts']:
        conid = contract['conid']
        if 'probability' not in contract:
            try:
                contract['probability'] = get_candidate_probability(conid)
            except:
                contract['probability'] = None
                print('error', contract['shortDescription'])
        try:
            print(conid, contract['shortDescription'], contract['probability']['probability_pct'])
        except:
            pass

  0%|          | 0/1 [00:00<?, ?it/s]

796056496 MNYCG Nov04'25 Sliwa YES @FORECASTX 5.0
796056501 MNYCG Nov04'25 Sliwa NO @FORECASTX 96.0
796056506 MNYCG Nov04'25 Cuomo YES @FORECASTX 16.0
796056511 MNYCG Nov04'25 Cuomo NO @FORECASTX 85.0
796056520 MNYCG Nov04'25 Mamdani YES @FORECASTX 68.0
796056525 MNYCG Nov04'25 Mamdani NO @FORECASTX 33.0
796056531 MNYCG Nov04'25 Adams YES @FORECASTX 8.0
796056534 MNYCG Nov04'25 Adams NO @FORECASTX 93.0


In [25]:
f2 = [x.copy() for x in forecastex]

In [26]:
for market in f2:
    market['contracts'] = [x for x in market['contracts'] if x['probability']]

In [27]:
f2 = [market for market in f2 if market['contracts']]

## Open Interest (must run chrome in debug mode)

https://www.perplexity.ai/search/how-do-i-do-this-request-in-py-LKWOZo0lRNegn9r7ALA1dw

In [30]:
from OI_to_integer import OI_to_integer

In [31]:
from run_get_OI import run_get_OI

In [32]:
for market in tqdm(f2):
    for contract in tqdm(market['contracts']):
        if 'OI' not in contract:
            conid = contract['conid']
            OI = await run_get_OI([conid])
            OI = list(OI.values())[0]
            contract['OI'] = OI_to_integer(OI)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

WebSocket opened, sending requests for 1 conids...
Received OI for conid 796056496: 68.3K
Received all expected OI results, closing WebSocket.
✅ Received OI results for all conids.
WebSocket closed with code=None, message=None
WebSocket opened, sending requests for 1 conids...
Received OI for conid 796056501: 68.3K
Received all expected OI results, closing WebSocket.
✅ Received OI results for all conids.
WebSocket opened, sending requests for 1 conids...
Received OI for conid 796056506: 27.8K
Received all expected OI results, closing WebSocket.
✅ Received OI results for all conids.
WebSocket closed with code=None, message=None
WebSocket closed with code=None, message=None
WebSocket opened, sending requests for 1 conids...
Received OI for conid 796056511: 27.8K
Received all expected OI results, closing WebSocket.
✅ Received OI results for all conids.
WebSocket closed with code=None, message=None
WebSocket opened, sending requests for 1 conids...
Received OI for conid 796056520: 1.20M
Re

## Collect and filter

In [34]:
rows = []
for market in f2:
    for ctx in market['contracts']:
        contract = ctx.copy()
        probability = contract['probability']
        contract['probability_pct'] = probability['probability_pct']
        contract['weekly_volume'] = sum(probability['volume'])
        contract['yesNo'] = 'YES' if contract['putOrCall'] == 'C' else 'NO'
        for x in ['putOrCall','popularityRank', 'expiration', 'lastTradeMillis', 'lastTradeTime','eventFixedPayout', 'commodityCode', 'strike', 'market',
                    'categories', 'expectedResolutionTime', 'expectedPayoutTime', 'timespecifierParam', 'sourceAgency', 'marketRulesLink', 
                    'exchange', 'priceIncrement', 'currency', 'timezone', 'eventAuthorityURL', 'probability', 'tradingHours']:
            del contract[x]
        rows.append(contract)

In [41]:
from add_category_to_markets import add_category_to_markets
df = add_category_to_markets(rows, 'longDescription')

In [37]:
df

,name,longDescription,lastTradeDate,underlyingName,conid,underlyingConid,underlyingSymbol,shortDescription,strikeLabel,OI,probability_pct,weekly_volume,yesNo,category
0,MNYCG Nov04'25 Sliwa,Will Curtis Sliwa win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056496,796056051,MNYCG,MNYCG Nov04'25 Sliwa YES @FORECASTX,Sliwa,68300.0,5.0,4130.0,YES,Election
1,MNYCG Nov04'25 Sliwa,Will Curtis Sliwa win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056501,796056051,MNYCG,MNYCG Nov04'25 Sliwa NO @FORECASTX,Sliwa,68300.0,96.0,4130.0,NO,Election
2,MNYCG Nov04'25 Cuomo,Will Andrew Cuomo win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056506,796056051,MNYCG,MNYCG Nov04'25 Cuomo YES @FORECASTX,Cuomo,27800.0,16.0,2775.0,YES,Election
3,MNYCG Nov04'25 Cuomo,Will Andrew Cuomo win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056511,796056051,MNYCG,MNYCG Nov04'25 Cuomo NO @FORECASTX,Cuomo,27800.0,85.0,2775.0,NO,Election
4,MNYCG Nov04'25 Mamdani,Will Zohran Mamdani win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056520,796056051,MNYCG,MNYCG Nov04'25 Mamdani YES @FORECASTX,Mamdani,1200000.0,68.0,40527.0,YES,Election
5,MNYCG Nov04'25 Mamdani,Will Zohran Mamdani win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056525,796056051,MNYCG,MNYCG Nov04'25 Mamdani NO @FORECASTX,Mamdani,1200000.0,33.0,40527.0,NO,Election
6,MNYCG Nov04'25 Adams,Will Eric Adams win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056531,796056051,MNYCG,MNYCG Nov04'25 Adams YES @FORECASTX,Adams,202000.0,8.0,10940.0,YES,Election
7,MNYCG Nov04'25 Adams,Will Eric Adams win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056534,796056051,MNYCG,MNYCG Nov04'25 Adams NO @FORECASTX,Adams,202000.0,93.0,10940.0,NO,Election


In [40]:
from datetime import datetime

In [43]:
market.keys()

dict_keys(['symbol', 'conid', 'name', 'contracts'])

In [99]:
df_yes = df[df.yesNo == 'YES']
df_yes

,name,longDescription,lastTradeDate,underlyingName,conid,underlyingConid,underlyingSymbol,shortDescription,strikeLabel,OI,probability_pct,weekly_volume,yesNo,category
0,MNYCG Nov04'25 Sliwa,Will Curtis Sliwa win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056496,796056051,MNYCG,MNYCG Nov04'25 Sliwa YES @FORECASTX,Sliwa,68300.0,5.0,4130.0,YES,Election
2,MNYCG Nov04'25 Cuomo,Will Andrew Cuomo win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056506,796056051,MNYCG,MNYCG Nov04'25 Cuomo YES @FORECASTX,Cuomo,27800.0,16.0,2775.0,YES,Election
4,MNYCG Nov04'25 Mamdani,Will Zohran Mamdani win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056520,796056051,MNYCG,MNYCG Nov04'25 Mamdani YES @FORECASTX,Mamdani,1200000.0,68.0,40527.0,YES,Election
6,MNYCG Nov04'25 Adams,Will Eric Adams win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056531,796056051,MNYCG,MNYCG Nov04'25 Adams YES @FORECASTX,Adams,202000.0,8.0,10940.0,YES,Election


In [105]:
bins = df_yes.apply(lambda row: f'{row.longDescription}', axis=1).values.tolist()

In [106]:
bins

['Will Curtis Sliwa win the New York City general election for mayor in 2025?',
 'Will Andrew Cuomo win the New York City general election for mayor in 2025?',
 'Will Zohran Mamdani win the New York City general election for mayor in 2025?',
 'Will Eric Adams win the New York City general election for mayor in 2025?']

In [107]:
bins = [{'props': {'title': bin, 'ai_title': '', 'order': 0, 'color': ''}, 'isActive': True, 'endDay': 0}
        for bin in bins]

In [108]:
ifp = {'id': market['conid'],
        'type': 'bins',
        'state': 'active',
        'dates': {'startDay': str(datetime.now())[0:10].replace('-',''),'endDay': contract['lastTradeDate']},
        'props': {'title': market['name'], 'shortTitle': '', 'details': ''},
        'kind': 'discrete',
        'bins': bins}

In [109]:
ifp

{'id': 796056051,
 'type': 'bins',
 'state': 'active',
 'dates': {'startDay': '20250729', 'endDay': '20251129'},
 'props': {'title': 'General Election for New York City Mayor',
  'shortTitle': '',
  'details': ''},
 'kind': 'discrete',
 'bins': [{'props': {'title': 'Will Curtis Sliwa win the New York City general election for mayor in 2025?',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0},
  {'props': {'title': 'Will Andrew Cuomo win the New York City general election for mayor in 2025?',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0},
  {'props': {'title': 'Will Zohran Mamdani win the New York City general election for mayor in 2025?',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
   'endDay': 0},
  {'props': {'title': 'Will Eric Adams win the New York City general election for mayor in 2025?',
    'ai_title': '',
    'order': 0,
    'color': ''},
   'isActive': True,
 

In [110]:
ifps = [ifp]

In [111]:
from save_ifps_to_disk import save_ifps_to_disk
id_to_ifp = save_ifps_to_disk(ifps)

In [59]:
from gather_news_for_ifps import gather_news_for_ifps
news = gather_news_for_ifps(ifps)

saved glimt/news/796056051.txt


In [60]:
from detailed_proposition import detailed_proposition
from wiki_semantic_search import wiki_semantic_search
from split_news_into_text_and_urls import split_news_into_text_and_urls
from create_source_summaries import create_source_summaries
from rephrase_binary_outcomes import rephrase_binary_outcomes
from format_research import format_research
from glimt_forecast_prompt import glimt_forecast_prompt
from humor_me import humor_me
from get_forecast_components import *
from median_forecast import median_forecast
from median_rationale import median_rationale
from datetime import datetime

loading massive wiki index 2025-07-29 12:28:22.153708
loading wiki article titles 2025-07-29 12:29:29.927931
loading sentence transformer model 2025-07-29 12:29:33.290405
done 2025-07-29 12:29:36.057917


In [61]:
for ifp in ifps:
    break

In [ ]:
    print('begin FORECASTING', ifp['id'], ifp['props']['title'], datetime.now())
    title_plus_criteria = detailed_proposition(ifp)
    wiki_articles = wiki_semantic_search(title_plus_criteria)
    ifp_news_sources, ifp_news_text = split_news_into_text_and_urls(ifp, news)

In [ ]:
    source_summaries, sources = create_source_summaries(ifp['id'], title_plus_criteria, wiki_articles, ifp_news_sources, ifp_news_text)
    rephrase_binary_outcomes(ifp)
    research = format_research(source_summaries)

In [118]:
import json
from saved_prompt import saved_prompt
from get_periods_of_whenwill_question import get_periods_of_whenwill_question
from filter_periods_to_today import filter_periods_to_today
from get_event import get_event
from is_election import is_election

def glimt_forecast_prompt(ifp, research):
    (fn, savep) = saved_prompt(ifp)
    if savep: return savep
    ## Prompt with rationale fields split into for and against
    details = ifp['props']['details']
    title = ifp['props']['title']
    bins = [x['props']['title'] for x in ifp['bins']]
    rejected = []
    if 'When' in title or "By what date" in title:
        periods = get_periods_of_whenwill_question(ifp)
        rejected, filtered = filter_periods_to_today(periods)
        original_bins = bins
        event = get_event(ifp)
        bins = [f"{event} between {b} and {c}" for a,b,c in filtered]
    sb1 = '\n'.join([f"""* O{i+1}. {bin}""" for i, bin in enumerate(bins)])
    psum = '+'.join([f'P{i+1}' for i, bin in enumerate(bins)])
    pcom = ','.join([f'P{i+1}' for i, bin in enumerate(bins)])
    sbins = f"""The question has one of {len(bins)} outcomes namely  

{sb1}

Each outcome Oi has a probability Pi where 0 <= Pi <= 1.
We must have that {psum} = 1.0.
Add some reasonable amount of randomness/noise to the estimation of the branches.
The output is a Python list wrapped by a binProbs tag, in this format:
```binProbs
[{pcom}]
```
"""
    prompt = f"""
You are a talented, experienced and confident superforecaster. You are asked a question:

```question
{title}
```

You are given details on how to interpret the terms of the question:

```details
{details}
```

Your assistant has research related news and Wikipedia articles and prepared summaries of each one.
Use the data in these research summaries to analyse the question:

{research}

For you to be marked Successful, you must output 3 things:

1. Probabilities for the outcomes of the question.  
{sbins}

2. Reasons your probabilities might be right, wrapped in tag in this format:
```rRight
...reasons you might be right
```
There should be a reason you are right for each outcome ending with a forecast of form ZZ% where ZZ ranges from 0 to 100.

3. Reasons your probabilities might be wrong, wrapped in tag in this format:
```rWrong
...reasons you might be wrong
```
"""
    with open(fn, 'w') as f:
        json.dump((prompt, rejected), f)
    fn1 = fn.replace('.json', '.txt')
    with open(fn1, 'w') as f:
        f.write(prompt)

    return prompt, rejected

In [120]:
    prompt, rejected = glimt_forecast_prompt(ifp, research)

In [ ]:
print(prompt)

'General Election for New York City Mayor'

In [135]:
prompt_polls = "Any news articles giving details of election polls for {market['name']}"

In [136]:
from call_asknews import call_asknews

In [157]:
polls = call_asknews(market['name'] + " Opinion Poll Results")

In [158]:
print(polls)

Here are the relevant news articles:

**The Upper House Election and the New York Mayoral Election: A Shift in the Political Landscape?**
The recent upper house election and New York mayoral election have raised questions about the existing political landscape. In the upper house election, the ruling Liberal Democratic Party and Komeito suffered a crushing defeat, while the Constitutional Democratic Party of Japan and the Reiwa Shinsengumi Party gained significant seats. Both parties focused on the issue of 'foreigners' in their campaigns, with the Reiwa Shinsengumi Party's 'Japanese First' slogan being particularly prominent. However, the reality is that the issue of 'foreigners' is not as significant as it seems, with only 3% of welfare recipients being foreigners. The Minister of Health, Labour and Welfare, Fukuda Kazuo, stated that there is no evidence of foreigners dominating medical expenses or receiving preferential treatment in welfare. In reality, the Constitutional Democratic

In [140]:
prompt2 = f"""{prompt}

Also take into account these articles discussing election polls for {market['name']}:

{polls}"""

In [143]:
with open('foo.txt', 'w') as f:
    f.write(prompt2)

In [148]:
!emacs foobar.txt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [150]:
with open('foobar.txt', 'r') as f:
    prompt2 = f.read()

In [151]:
    ## Run the prompt 5 times
    prompt_tries = 1 # Waste of time on Mistral 4 bit
    answers = [humor_me(prompt2, i+1) for i in range(prompt_tries)]

START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.10967367092768351

```binProbs
[0.25, 0.20, 0.40, 0.15]
```

```rRight
O1. Curtis Sliwa has a strong Republican base and is competitive in recent polling, which suggests he could win the election. His support is around 10-22%, and with the right campaign developments, he could consolidate more votes. Forecast: 25%.

O2. Andrew Cuomo is running as an independent and has a viable third-party challenge, with support around 23-29%. His experience and name recognition could help him win, especially if he can attract more undecided voters. Forecast: 20%.

O3. Zohran Mamdani is the Democratic nominee and leads in most recent polls with support around 26-41%. His progressive platform and consolidation of Democratic support make him the frontrunner. Forecast: 40%.

O4. Eric Adams, the incumbent, is trailing with poor approval ratings and support around 13-16%. His chances appear slim 

In [152]:
df_yes

,name,longDescription,lastTradeDate,underlyingName,conid,underlyingConid,underlyingSymbol,shortDescription,strikeLabel,OI,probability_pct,weekly_volume,yesNo,category
0,MNYCG Nov04'25 Sliwa,Will Curtis Sliwa win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056496,796056051,MNYCG,MNYCG Nov04'25 Sliwa YES @FORECASTX,Sliwa,68300.0,5.0,4130.0,YES,Election
2,MNYCG Nov04'25 Cuomo,Will Andrew Cuomo win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056506,796056051,MNYCG,MNYCG Nov04'25 Cuomo YES @FORECASTX,Cuomo,27800.0,16.0,2775.0,YES,Election
4,MNYCG Nov04'25 Mamdani,Will Zohran Mamdani win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056520,796056051,MNYCG,MNYCG Nov04'25 Mamdani YES @FORECASTX,Mamdani,1200000.0,68.0,40527.0,YES,Election
6,MNYCG Nov04'25 Adams,Will Eric Adams win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056531,796056051,MNYCG,MNYCG Nov04'25 Adams YES @FORECASTX,Adams,202000.0,8.0,10940.0,YES,Election


In [ ]:
    binProbs = [get_bin_probs(a) for a in answers]
    rights = [get_rights(a) for a in answers]
    wrongs = [get_wrongs(a) for a in answers]
    ## Median forecasts and rationales
    forecast = rejected + median_forecast(binProbs)
    right = median_rationale(rights)
    wrong = median_rationale(wrongs)
    result = (forecast, right, wrong, sources)
    fn = f'glimt/forecast'
    import os
    os.makedirs(fn, exist_ok=True)
    fn = f"{fn}/{ifp['id']}.json"
    import json
    with open(fn, 'w') as f:
        json.dump(result, f)

In [70]:
df

,name,longDescription,lastTradeDate,underlyingName,conid,underlyingConid,underlyingSymbol,shortDescription,strikeLabel,OI,probability_pct,weekly_volume,yesNo,category
0,MNYCG Nov04'25 Sliwa,Will Curtis Sliwa win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056496,796056051,MNYCG,MNYCG Nov04'25 Sliwa YES @FORECASTX,Sliwa,68300.0,5.0,4130.0,YES,Election
1,MNYCG Nov04'25 Sliwa,Will Curtis Sliwa win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056501,796056051,MNYCG,MNYCG Nov04'25 Sliwa NO @FORECASTX,Sliwa,68300.0,96.0,4130.0,NO,Election
2,MNYCG Nov04'25 Cuomo,Will Andrew Cuomo win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056506,796056051,MNYCG,MNYCG Nov04'25 Cuomo YES @FORECASTX,Cuomo,27800.0,16.0,2775.0,YES,Election
3,MNYCG Nov04'25 Cuomo,Will Andrew Cuomo win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056511,796056051,MNYCG,MNYCG Nov04'25 Cuomo NO @FORECASTX,Cuomo,27800.0,85.0,2775.0,NO,Election
4,MNYCG Nov04'25 Mamdani,Will Zohran Mamdani win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056520,796056051,MNYCG,MNYCG Nov04'25 Mamdani YES @FORECASTX,Mamdani,1200000.0,68.0,40527.0,YES,Election
5,MNYCG Nov04'25 Mamdani,Will Zohran Mamdani win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056525,796056051,MNYCG,MNYCG Nov04'25 Mamdani NO @FORECASTX,Mamdani,1200000.0,33.0,40527.0,NO,Election
6,MNYCG Nov04'25 Adams,Will Eric Adams win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056531,796056051,MNYCG,MNYCG Nov04'25 Adams YES @FORECASTX,Adams,202000.0,8.0,10940.0,YES,Election
7,MNYCG Nov04'25 Adams,Will Eric Adams win the New York City general election for mayor in 2025?,20251129,General Election for New York City Mayor,796056534,796056051,MNYCG,MNYCG Nov04'25 Adams NO @FORECASTX,Adams,202000.0,93.0,10940.0,NO,Election


In [72]:
[x['props']['title'] for x in ifp['bins']]

['The following resolves as YES: Will Curtis Sliwa win the New York City general election for mayor in 2025?. Last trade date 20251129. Contract ID: 796056496',
 'The following resolves as NO: Will Curtis Sliwa win the New York City general election for mayor in 2025?. Last trade date 20251129. Contract ID: 796056501',
 'The following resolves as YES: Will Andrew Cuomo win the New York City general election for mayor in 2025?. Last trade date 20251129. Contract ID: 796056506',
 'The following resolves as NO: Will Andrew Cuomo win the New York City general election for mayor in 2025?. Last trade date 20251129. Contract ID: 796056511',
 'The following resolves as YES: Will Zohran Mamdani win the New York City general election for mayor in 2025?. Last trade date 20251129. Contract ID: 796056520',
 'The following resolves as NO: Will Zohran Mamdani win the New York City general election for mayor in 2025?. Last trade date 20251129. Contract ID: 796056525',
 'The following resolves as YES: 

In [74]:
forecast[0][::2]

[0.1, 0.05, 0.05, 0.2]

In [66]:
contracts = df.to_dict(orient='records')

In [67]:
contracts = {contract['conid']: contract for contract in contracts}

In [68]:
for ifp in ifps:
    fn = f'glimt/forecast'
    fn = f"{fn}/{ifp['id']}.json"
    with open(fn, 'r') as f:
        forecast = json.load(f)
    [[Pyes, Pno], Rfor, Ragainst, sources] = forecast
    contract = contracts[ifp['id']]
    yesNo = contract['yesNo']
    pct = contract['probability_pct']
    yesPct = pct if yesNo == 'YES' else 100-pct
    noPct = 100-pct if yesNo == 'YES' else pct
    report = f"""
{ifp['props']['title']}
{ifp['props']['shortTitle']}

YES {100*Pyes} bot / {yesPct} crowd
NO  {100*Pno} bot  / {noPct} crowd

Reasons for YES
===============
{Rfor}

Reasosn for NO
==============
{Ragainst}

----------------------------------------"""
    print(report)

ValueError: too many values to unpack (expected 2)